
# Practical Lab 2: Multivariate Linear Regression, Non-Parametric Models and Cross-Validation

### Name: John Buni 
### Student ID: 9115726

## Part 1 — Data Exploration and Preparation

### 1.1 Introduction and Problem Framing

This lab investigates the prediction of diabetes disease progression
one year after baseline using the Scikit-Learn Diabetes Dataset.

The dataset contains 442 observations, ten baseline predictor variables,
and a continuous target representing disease progression.

This is a supervised regression problem. Unlike classification, which
predicts discrete categories, regression predicts numerical outcomes.

The lab compares polynomial regression and non-parametric models,
including decision trees and k-nearest neighbors.

The analysis uses R², Mean Absolute Error (MAE), and Mean Absolute
Percentage Error (MAPE) to evaluate regression performance.

We will use a 75% training, 10% validation, and 15% testing split.
The validation set will guide model selection, and the test set will
remain untouched until final evaluation.


In [ ]:
from IPython.display import display
from src.diabetes_data import DiabetesData

# Initialize the object-oriented data manager.
diabetes_data = DiabetesData()

# Load the dataset.
df = diabetes_data.load_data()

print("Dataset shape:", diabetes_data.get_shape())
print("\nPredictor variables:")
print(diabetes_data.get_feature_names())

print("\nFirst five observations:")
display(df.head())

print("\nDescriptive statistics:")
display(diabetes_data.get_summary())

print("\nMissing values by column:")
display(diabetes_data.get_missing_values())



### 1.2 Exploratory Data Analysis

Exploratory Data Analysis (EDA) helps identify patterns, relationships,
and potential data-quality issues before model training.

We will examine:

- The distributions of the predictor variables and target.
- Relationships between BMI, blood pressure, and disease progression.
- Correlations between variables to identify potentially useful predictors
  and possible multicollinearity.

These findings will help guide feature selection and model development.


In [ ]:

from src.data_visualizer import DataVisualizer

# Create the visualization object.
visualizer = DataVisualizer(df)

# Explore variable distributions.
visualizer.plot_histograms()

# Examine relationships with the target.
visualizer.plot_scatter("bmi")
visualizer.plot_scatter("bp")

# Examine correlations between variables.
visualizer.plot_correlation_matrix()


### 1.3 EDA Findings and Interpretation

**Variable distributions:** The histograms show differences in the distributions of the baseline measurements and disease progression scores. The predictors are centered near zero because Scikit-Learn provides standardized measurements. The target remains on its original numerical scale.

**BMI and disease progression:** The BMI scatterplot shows a generally positive association between standardized BMI and diabetes progression. However, considerable variation remains among individual observations, suggesting that BMI alone cannot fully explain progression outcomes.

**Blood pressure and disease progression:** The blood-pressure scatterplot also suggests a positive relationship with the target, although individual observations vary considerably.

**Correlations:** The correlation matrix identifies BMI and s5 as particularly relevant predictors. It also shows relationships among predictor variables, which may affect the interpretation of multivariate regression coefficients.

Overall, the exploratory findings support comparing simple and more flexible models to determine whether additional predictors and nonlinear relationships improve predictive performance.

The scikit-learn Diabetes predictors are mean-centered and scaled, rather than provided as raw measurements. Their standard deviations are not necessarily equal to one, so the supplied values should not be interpreted as conventional z-scores.

In [ ]:
import importlib
import src.diabetes_data as diabetes_module

# Reload the updated class without restarting the notebook.
importlib.reload(diabetes_module)

diabetes_data = diabetes_module.DiabetesData()
df = diabetes_data.load_data()

print("Predictor correlations with disease progression:")
display(diabetes_data.get_target_correlations().round(3))



### 1.4 Data Quality and Cleaning Decisions

The dataset contains 442 observations, ten predictor variables,
and one continuous target variable.

The missing-value inspection found no missing observations, so
imputation is not required.

The predictor variables are already standardized by Scikit-Learn.
Therefore, additional feature scaling is not necessary for the
initial experiments.

No observations have been removed at this stage. Potentially unusual
values will be retained unless there is evidence that they are invalid,
because removing legitimate observations could distort the analysis.

Any additional preprocessing required by a model will be fitted using
training data only to prevent data leakage.


In [ ]:
print("Duplicate rows:", diabetes_data.get_duplicate_count())

The duplicate-row inspection identified zero fully duplicated observations. Therefore, no duplicate removal was necessary.

### 1.5 Correlation Findings

The correlation analysis identified BMI (r = 0.586), s5 (r = 0.566), and blood pressure (r = 0.441) as the three predictors most strongly associated with diabetes disease progression in absolute terms. All three show positive linear relationships with the target.

The s3 variable showed a negative correlation (r = −0.395), suggesting an inverse linear association with disease progression.

BMI had the strongest individual linear correlation with the target, supporting its selection for the univariate polynomial regression experiments in Part 2.

However, correlation does not establish causation, and a predictor's individual correlation does not necessarily indicate how useful it will be in a multivariate model. Correlations between predictors may also affect model interpretation.

These findings are exploratory and will be assessed further using held-out validation and test data.

### 1.6 Training, Validation, and Testing Split

The dataset is divided into approximately 75% training, 10% validation,
and 15% testing data using a fixed random seed for reproducibility.

The training set is used to fit models, the validation set is used
to select models and hyperparameters, and the test set is reserved
for final evaluation.

Keeping the test set separate helps reduce overly optimistic
performance estimates caused by repeated model selection.

In [ ]:

import importlib
import src.diabetes_data as diabetes_module

importlib.reload(diabetes_module)

diabetes_data = diabetes_module.DiabetesData()
df = diabetes_data.load_data()

splits = diabetes_data.split_data(random_state=42)

X_train = splits["X_train"]
X_val = splits["X_val"]
X_test = splits["X_test"]

y_train = splits["y_train"]
y_val = splits["y_val"]
y_test = splits["y_test"]

print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Testing samples:    {len(X_test)}")
print(f"Total samples:      {len(X_train) + len(X_val) + len(X_test)}")


In [ ]:

# Verify that the three datasets contain different observations.
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

# Verify that every original observation is accounted for.
assert len(X_train) + len(X_val) + len(X_test) == len(df)

print("All split integrity checks passed.")


### 1.7 Part 1 Summary

The Scikit-Learn Diabetes Dataset was successfully loaded and explored using reusable object-oriented Python classes.

Exploratory analysis found no missing values and identified BMI (r = 0.586), s5 (r = 0.566), and blood pressure (r = 0.441) as the predictors with the strongest individual linear relationships with disease progression.

The predictor variables are already standardized, and no observations were removed during initial preparation.

The dataset was divided into 330 training observations (74.66%), 45 validation observations (10.18%), and 67 testing observations (15.16%). The split uses a fixed random seed to ensure reproducibility.

The training and validation datasets will support model fitting and selection. The testing dataset will remain untouched until final model evaluation.

These preparations establish a reproducible foundation for comparing machine learning models in the following sections.


## Part 2 — Univariate Polynomial Regression Using BMI

### 2.1 Objective and Methodology

In this section, we investigate whether BMI alone can predict
diabetes disease progression one year after baseline.

BMI was selected because it showed the strongest individual linear
correlation with the target during exploratory analysis (r = 0.586).

We will train polynomial regression models with degrees 0 through 5.

- Degree 0 represents a constant baseline model.
- Degree 1 represents ordinary linear regression.
- Degrees 2 through 5 introduce increasingly flexible polynomial terms.

Each model will be trained using only the training dataset.

We will compare training and validation performance using R²,
Mean Absolute Error (MAE), and Mean Absolute Percentage Error (MAPE).

The polynomial degree will be selected using validation performance.
The test dataset will remain untouched until a final model is chosen.

Because the BMI predictor is standardized, its values should not be
interpreted as conventional BMI measurements.


In [ ]:

import importlib
import src.models as models_module

# Reload our updated object-oriented model class.
importlib.reload(models_module)

PolynomialRegressionModel = models_module.PolynomialRegressionModel

# Use BMI as our only predictor.
X_train_bmi = X_train[["bmi"]]
X_val_bmi = X_val[["bmi"]]
X_test_bmi = X_test[["bmi"]]

# Train polynomial models of degrees 0 through 5.
polynomial_models = {}

for degree in range(6):
    model = PolynomialRegressionModel(degree=degree)
    model.fit(X_train_bmi, y_train)

    polynomial_models[degree] = model

    print(f"Degree {degree}: model trained successfully.")

print(f"\nTotal models trained: {len(polynomial_models)}")



### 2.2 Training and Validation Performance

We evaluate polynomial regression models of degrees 0 through 5
using three metrics:

- **R² (Coefficient of Determination):** Measures how much of the
  variation in diabetes progression is explained by the model.
  Higher values are better.
- **MAE (Mean Absolute Error):** Measures the average absolute
  difference between actual and predicted progression values.
  Lower values are better.
- **MAPE (Mean Absolute Percentage Error):** Expresses prediction
  errors as a percentage of the actual target values.
  Lower values are better.

Training performance helps us understand how well each model fits
the observed training data. Validation performance helps us assess
generalization to unseen observations.

The test set is reserved for the final selected model.


In [ ]:

from src.evaluation import ModelEvaluator
import pandas as pd

evaluator = ModelEvaluator(polynomial_models)

train_results = evaluator.evaluate_regression(
    X_train_bmi, y_train, "Training"
)

val_results = evaluator.evaluate_regression(
    X_val_bmi, y_val, "Validation"
)

comparison_results = pd.concat(
    [train_results, val_results],
    ignore_index=True
)

display(
    comparison_results
    .sort_values(["Degree", "Dataset"])
    .round(3)
)



### 2.3 Model Comparison and Selection

The degree 0 baseline achieved approximately zero R², indicating
that a constant prediction does not explain variation in disease
progression.

Polynomial models of degrees 1 through 5 performed substantially
better, with validation R² values between approximately 0.447
and 0.454.

The differences between these models were relatively small.
Degree 3 achieved the lowest validation MAE (41.741), while
degree 4 achieved the lowest validation MAPE (40.241%).

For model selection, validation R² was chosen as the primary
criterion because it measures the proportion of variation
explained by the regression model.

Degree 5 achieved the highest validation R² (0.454) and was
therefore selected as the final polynomial regression model.

Although training performance improved slightly as polynomial
degree increased, the validation results do not show a substantial
benefit from greater complexity. The results also do not provide
strong evidence of severe overfitting.

The test dataset has not been used during model selection.


In [ ]:

# Select the polynomial degree with the highest validation R².
best_row = val_results.loc[val_results["R²"].idxmax()]

best_degree = int(best_row["Degree"])
best_model = polynomial_models[best_degree]

print(f"Selected polynomial degree: {best_degree}")
print(f"Validation R²: {best_row['R²']:.3f}")
print(f"Validation MAE: {best_row['MAE']:.3f}")
print(f"Validation MAPE: {best_row['MAPE (%)']:.3f}%")



### 2.4 Final Test Evaluation

The degree 5 polynomial regression model was selected because it
achieved the highest validation R².

To estimate how well the selected model generalizes to unseen data,
we now evaluate it on the held-out test dataset.

The test set was not used for training, model comparison, or
polynomial degree selection.

We report R², MAE, and MAPE using the same evaluation metrics
applied to the training and validation datasets.


In [ ]:

# Evaluate only the selected model on the held-out test set.
final_evaluator = ModelEvaluator({
    best_degree: best_model
})

test_results = final_evaluator.evaluate_regression(
    X_test_bmi,
    y_test,
    "Testing"
)

display(test_results.round(3))


In [ ]:

# Compare the selected model's validation and test results.
final_comparison = pd.concat([
    val_results[val_results["Degree"] == best_degree],
    test_results
], ignore_index=True)

display(final_comparison.round(3))



### 2.5 Visualization of the Selected Polynomial Model

The following figure displays the relationship between standardized
BMI and diabetes progression, together with the fitted degree 5
polynomial regression curve.

Training, validation, and test observations are shown separately.

The curve was fitted using training data only. Validation data
was used to select the polynomial degree, while test data was
reserved for final evaluation.

The visualization helps assess whether the polynomial captures
the general trend and whether additional complexity introduces
unusual curvature.


In [ ]:

import importlib
import src.data_visualizer as visualizer_module

# Reload the updated class.
importlib.reload(visualizer_module)

DataVisualizer = visualizer_module.DataVisualizer

visualizer = DataVisualizer(df)

visualizer.plot_polynomial_regression(
    model=best_model,
    X_train=X_train_bmi,
    y_train=y_train,
    X_val=X_val_bmi,
    y_val=y_val,
    X_test=X_test_bmi,
    y_test=y_test,
    degree=best_degree
)



### 2.6 Fitted Polynomial Equation and Model Parameters

The selected degree 5 polynomial regression model represents
diabetes progression as a function of standardized BMI.

The general polynomial equation is:

ŷ = β₀ + β₁x + β₂x² + β₃x³ + β₄x⁴ + β₅x⁵

where:

- ŷ is the predicted diabetes progression.
- x is standardized BMI.
- β₀ is the intercept.
- β₁ through β₅ are the fitted polynomial coefficients.

The fitted coefficients will be extracted from the trained model
and displayed to two decimal places.


In [ ]:

# Extract the fitted model parameters.
intercept = best_model.get_intercept()
coefficients = best_model.get_coefficients()
feature_names = best_model.get_feature_names(["bmi"])

print(f"Selected degree: {best_degree}")
print(f"Intercept: {intercept:.2f}")

print("\nPolynomial terms and coefficients:")
for feature, coefficient in zip(feature_names, coefficients):
    print(f"{feature}: {coefficient:.2f}")

# Construct the fitted equation to two decimal places.
equation = f"ŷ = {intercept:.2f}"

for feature, coefficient in zip(feature_names, coefficients):
    equation += f" {coefficient:+.2f}({feature})"

print("\nFitted polynomial equation:")
print(equation)


The fitted degree 5 polynomial regression equation contains an intercept of 155.08 and five polynomial coefficients. The coefficients for the higher-degree terms are relatively large because the standardized BMI values are small and their higher powers approach zero. The model's curved shape, particularly at the lower end of the BMI range, suggests that its predictions may be sensitive to extreme BMI values. Therefore, the additional flexibility of the degree 5 model should be interpreted cautiously.

In [ ]:

# Count the fitted parameters, including the intercept.
feature_names = best_model.get_feature_names(["bmi"])

n_polynomial_terms = len(feature_names)
n_parameters = n_polynomial_terms + 1

print("Polynomial features:", feature_names)
print("Number of polynomial terms:", n_polynomial_terms)
print("Intercept parameters: 1")
print("Total fitted parameters:", n_parameters)



The selected degree 5 model has six fitted parameters:
one intercept and five polynomial coefficients.

Scikit-learn's PolynomialFeatures.get_feature_names_out()
identifies the five generated features: bmi, bmi², bmi³,
bmi⁴, and bmi⁵.

Because LinearRegression fits the intercept separately,
the total number of parameters is 5 + 1 = 6.

Increasing the polynomial degree increases model flexibility,
but it can also increase sensitivity to the training data.
The curvature observed at the lower end of the BMI range
illustrates why higher-degree models should be interpreted
carefully.


In [ ]:

# Compare trainable parameter counts for all polynomial degrees.
parameter_rows = []

for degree, model in polynomial_models.items():
    feature_names = model.get_feature_names(["bmi"])

    # Degree 0 is an intercept-only model.
    n_parameters = 1 if degree == 0 else len(feature_names) + 1

    parameter_rows.append({
        "Polynomial Degree": degree,
        "Trainable Parameters": n_parameters
    })

parameter_counts = pd.DataFrame(parameter_rows)

display(parameter_counts.sort_values("Polynomial Degree"))



The number of trainable parameters increases by one for
each additional polynomial degree.

Degree 0 fits only a constant intercept, while degree 1
fits an intercept and one BMI coefficient.

Each higher degree adds another polynomial coefficient.
Therefore, a degree d univariate polynomial model has
d + 1 trainable parameters, including the intercept.

Although higher-degree models are more flexible, the
additional parameters may increase the risk of overfitting.



### 2.7 Predicting Disease Progression from BMI

The selected polynomial regression model can predict disease
progression for a new standardized BMI value.

We use the model's predict() method to generate a prediction.

Because the diabetes dataset contains standardized BMI values,
the input must be expressed on the same standardized scale
used during training.


In [ ]:

# Example prediction using a standardized BMI value.
example_bmi = 0.05

new_observation = pd.DataFrame({
    "bmi": [example_bmi]
})

predicted_progression = best_model.predict(new_observation)[0]

print(f"Standardized BMI: {example_bmi:.2f}")
print(f"Predicted disease progression: {predicted_progression:.2f}")



For a standardized BMI value of 0.05, the selected degree 5
polynomial regression model predicts a diabetes progression
score of 202.46.

This prediction demonstrates how the trained model can estimate
disease progression for a new observation using BMI alone.

However, the prediction is subject to uncertainty because the
model does not incorporate other potentially relevant predictors.




### 2.8 Part 2 Summary and Conclusion

Six polynomial regression models, ranging from degree 0 to
degree 5, were trained using standardized BMI as the sole
predictor of diabetes progression.

Degree 5 was selected because it achieved the highest
validation R² of 0.454.

On the held-out test dataset, the selected model achieved:

- R² = 0.197
- MAE = 54.548
- MAPE = 47.884%

The lower test R² compared with validation R² indicates that
the model generalized less effectively to the test observations
than expected from validation performance.

The fitted polynomial curve showed curvature at the lower
end of the BMI range, highlighting the flexibility and
potential instability of higher-degree polynomial models.

The selected model contains six fitted parameters: one
intercept and five polynomial coefficients.

For a standardized BMI of 0.05, the model predicted a
disease progression score of 202.46.

Overall, BMI provides useful predictive information, but
BMI alone is insufficient to explain all variation in
diabetes progression. The multivariate models in Part 3
will investigate whether additional predictors improve
performance.



### 2.9 Model Failures and Limitations

The selected degree 5 BMI-only polynomial regression model
achieved a validation R² of 0.454 but a lower test R² of
0.197. This indicates weaker performance on previously
unseen test observations.

The model has several limitations:

1. **Single predictor:** BMI alone cannot capture all
   factors associated with diabetes progression.

2. **High polynomial degree:** Degree 5 introduces
   additional flexibility, but the higher-order terms
   may produce unstable predictions, especially near
   the extremes of the observed BMI range.

3. **Prediction errors:** The test MAE of 54.548 means
   that predictions differ from actual progression scores
   by approximately 54.55 units on average. The test
   MAPE of 47.884% indicates substantial relative error.

4. **Limited sample size:** The dataset contains only
   442 observations, so validation and test performance
   may depend on the particular data split.

5. **Clinical applicability:** The target represents
   a continuous disease-progression score, not a
   diabetes diagnosis or a calibrated risk probability.
   The model is not validated for clinical screening.

Overall, the selected polynomial model captures some
relationship between BMI and diabetes progression but
has limited predictive reliability.

The results motivate the use of additional predictors
and alternative modeling approaches in Part 3.



## Part 3 — Multivariate and Non-Parametric Models

### 3.1 Multivariate Polynomial Regression

In this section, we investigate whether using all ten predictors
improves the prediction of diabetes disease progression compared
with the univariate BMI-only model from Part 2.

We train two multivariate polynomial regression models:

- Degree 2
- Degree 3

Polynomial feature expansion introduces squared terms and
interactions between predictors, allowing the models to capture
more complex relationships.

Both models use the original training and validation datasets
from Part 1. The held-out test dataset remains separate from
model selection.

We evaluate regression performance using R², MAE, and MAPE.

Higher polynomial degrees generate substantially more features,
which can increase model complexity and the risk of overfitting.


In [ ]:

import importlib
import src.models as models_module

# Reload the updated model definitions.
importlib.reload(models_module)

MultivariatePolynomialRegressionModel = (
    models_module.MultivariatePolynomialRegressionModel
)

# Use all ten predictors from Part 1.
multivariate_models = {}

for degree in [2, 3]:
    model = MultivariatePolynomialRegressionModel(degree=degree)
    model.fit(X_train, y_train)

    multivariate_models[degree] = model

    print(f"Degree {degree}: trained successfully.")
    print(
        "Generated polynomial features:",
        len(model.get_feature_names(X_train.columns.tolist()))
    )
    print()



### 3.2 Multivariate Polynomial Model Evaluation

The degree 2 and degree 3 multivariate polynomial models
are evaluated using the training and validation datasets.

We use the same regression metrics as Part 2:

- R²: Higher values indicate better explanatory performance.
- MAE: Lower values indicate smaller average absolute errors.
- MAPE: Lower values indicate smaller relative prediction errors.

Comparing training and validation performance helps identify
potential overfitting.

The degree 3 model generates 285 polynomial features compared
with 65 for degree 2. Because the training dataset contains
330 observations, the degree 3 model may be particularly
susceptible to overfitting.

The test dataset is not used in this comparison.


In [ ]:

# Evaluate multivariate polynomial regression models.
multivariate_evaluator = ModelEvaluator(multivariate_models)

multivariate_train_results = multivariate_evaluator.evaluate_regression(
    X_train,
    y_train,
    "Training"
)

multivariate_val_results = multivariate_evaluator.evaluate_regression(
    X_val,
    y_val,
    "Validation"
)

# Combine training and validation results.
multivariate_comparison = pd.concat(
    [multivariate_train_results, multivariate_val_results],
    ignore_index=True
)

display(
    multivariate_comparison
    .sort_values(["Degree", "Dataset"])
    .round(3)
)



### 3.3 Multivariate Polynomial Regression Findings

The degree 2 multivariate polynomial regression model achieved
a training R² of 0.598 and validation R² of 0.603, indicating
relatively consistent performance across both datasets.

Its validation MAE was 37.745, compared with 42.083 for the
selected BMI-only polynomial model in Part 2.

This suggests that using all ten predictors provides more useful
predictive information than BMI alone.

In contrast, the degree 3 multivariate model achieved a high
training R² of 0.891 but a severely negative validation R²
of -125.339. Its validation MAE increased to 333.915.

This large difference between training and validation performance
indicates severe overfitting and poor generalization.

The degree 3 model generates 285 polynomial features from only
330 training observations, which may contribute to unstable
coefficient estimates.

Based on validation R² and MAE, the degree 2 model is preferred
over the degree 3 model.

These findings demonstrate that increasing model complexity
does not necessarily improve performance on unseen data.

**Numerical stability note:** The degree-3 model expands the ten original predictors into 285 polynomial features. With only 330 training observations, the resulting regression problem may be numerically ill-conditioned. Its coefficients and evaluation metrics can therefore vary across software environments due to differences in numerical precision and linear algebra implementations. However, the main conclusion remains unchanged: degree 3 generalizes substantially worse than degree 2.


In [ ]:

# Select the multivariate polynomial model with the highest validation R².
best_multivariate_row = multivariate_val_results.loc[
    multivariate_val_results["R²"].idxmax()
]

best_multivariate_degree = int(best_multivariate_row["Degree"])
best_multivariate_model = multivariate_models[best_multivariate_degree]

print(f"Selected multivariate degree: {best_multivariate_degree}")
print(f"Validation R²: {best_multivariate_row['R²']:.3f}")
print(f"Validation MAE: {best_multivariate_row['MAE']:.3f}")
print(f"Validation MAPE: {best_multivariate_row['MAPE (%)']:.3f}%")



### 3.4 Decision Tree Regression

Decision tree regression is a non-parametric machine learning
method that predicts continuous outcomes by recursively splitting
the feature space into smaller regions.

Unlike polynomial regression, decision trees do not assume a
specific mathematical relationship between predictors and
the target.

We train two decision tree regression models using all ten
predictors:

- Model 1: Maximum depth = 3
- Model 2: Maximum depth = 8

The maximum depth controls model complexity. A shallow tree
may underfit the data, while a deeper tree may capture more
complex relationships but also overfit.

Both models use the same training and validation splits from
Part 1. Their performance will be compared using R², MAE,
and MAPE.


In [ ]:

import importlib
import src.models as models_module

# Reload the updated model definitions.
importlib.reload(models_module)

DecisionTreeRegressionModel = models_module.DecisionTreeRegressionModel

# Train two decision tree models.
tree_models = {}

for depth in [3, 8]:
    model = DecisionTreeRegressionModel(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)
    tree_models[depth] = model

    print(
        f"Max depth {depth}: "
        f"actual depth = {model.get_depth()}, "
        f"leaf nodes = {model.get_leaf_count()}"
    )

# Evaluate training and validation performance.
tree_evaluator = ModelEvaluator(tree_models)

tree_train_results = tree_evaluator.evaluate_regression(
    X_train, y_train, "Training"
)

tree_val_results = tree_evaluator.evaluate_regression(
    X_val, y_val, "Validation"
)

tree_comparison = pd.concat(
    [tree_train_results, tree_val_results],
    ignore_index=True
)

# The evaluator uses "Degree" as its generic model identifier.
# Rename it to reflect the decision tree hyperparameter.
tree_comparison = tree_comparison.rename(
    columns={"Degree": "Max Depth"}
)

display(
    tree_comparison
    .sort_values(["Max Depth", "Dataset"])
    .round(3)
)



### 3.5 Decision Tree Regression Findings

The decision tree with maximum depth 3 achieved a training R²
of 0.533 and validation R² of 0.413. Its validation MAE was
42.078, indicating moderate predictive performance.

The deeper tree with maximum depth 8 achieved a much higher
training R² of 0.912. However, its validation R² decreased
to 0.017, while its validation MAE increased to 57.707.

This substantial difference between training and validation
performance indicates that the depth 8 model overfits the
training dataset.

The depth 3 tree generalizes better because its limited
complexity reduces its ability to memorize training observations.

Based on validation R², MAE, and MAPE, the depth 3 decision
tree is preferred over the depth 8 model.

However, its validation R² of 0.413 is lower than the 0.603
achieved by the degree 2 multivariate polynomial regression
model. Therefore, the degree 2 polynomial model currently
provides better validation performance.


In [ ]:

# Select the decision tree with the highest validation R².
best_tree_row = tree_val_results.loc[
    tree_val_results["R²"].idxmax()
]

best_tree_depth = int(best_tree_row["Degree"])
best_tree_model = tree_models[best_tree_depth]

print(f"Selected decision tree depth: {best_tree_depth}")
print(f"Validation R²: {best_tree_row['R²']:.3f}")
print(f"Validation MAE: {best_tree_row['MAE']:.3f}")
print(f"Validation MAPE: {best_tree_row['MAPE (%)']:.3f}%")



### 3.6 k-Nearest Neighbors Regression

k-Nearest Neighbors (kNN) regression is a non-parametric method
that predicts continuous outcomes using the average target
value of nearby training observations.

We train two kNN regression models using all ten predictors:

- Model 1: k = 3
- Model 2: k = 15

The value of k controls how many neighboring observations
contribute to each prediction.

A smaller k can capture local patterns but may be sensitive
to noise and overfit the training data. A larger k produces
smoother predictions but may underfit.

Feature scaling is included in the model pipeline because
kNN uses distances to identify neighboring observations.

Both models are evaluated using training and validation R²,
MAE, and MAPE. The test dataset is not used for model selection.


In [ ]:

import importlib
import src.models as models_module

# Reload the updated model definitions.
importlib.reload(models_module)

KNNRegressionModel = models_module.KNNRegressionModel

# Train two kNN regression models.
knn_models = {}

for k in [3, 15]:
    model = KNNRegressionModel(n_neighbors=k)
    model.fit(X_train, y_train)

    knn_models[k] = model
    print(f"kNN model with k={k} trained successfully.")

# Evaluate both models.
knn_evaluator = ModelEvaluator(knn_models)

knn_train_results = knn_evaluator.evaluate_regression(
    X_train, y_train, "Training"
)

knn_val_results = knn_evaluator.evaluate_regression(
    X_val, y_val, "Validation"
)

knn_comparison = pd.concat(
    [knn_train_results, knn_val_results],
    ignore_index=True
)

# Rename the generic model identifier.
knn_comparison = knn_comparison.rename(
    columns={"Degree": "Neighbors (k)"}
)

display(
    knn_comparison
    .sort_values(["Neighbors (k)", "Dataset"])
    .round(3)
)



### 3.7 kNN Regression Findings

The kNN model with k=3 achieved a training R² of 0.641
but a lower validation R² of 0.323. Its validation MAE
was 50.526, suggesting that the model may overfit
the training observations.

The kNN model with k=15 achieved a training R² of 0.489
and validation R² of 0.553. Its validation MAE was
40.188, and its validation MAPE was 41.171%.

Although k=15 has lower training performance, it
generalizes better to unseen validation observations.

Based on validation R², MAE, and MAPE, the kNN model
with k=15 is selected.

Compared with the other regression models evaluated
so far, k=15 performs better than the selected
decision tree but does not outperform the degree 2
multivariate polynomial regression model.


In [ ]:

# Select the kNN model with the highest validation R².
best_knn_row = knn_val_results.loc[
    knn_val_results["R²"].idxmax()
]

best_knn_k = int(best_knn_row["Degree"])
best_knn_model = knn_models[best_knn_k]

print(f"Selected kNN neighbors: {best_knn_k}")
print(f"Validation R²: {best_knn_row['R²']:.3f}")
print(f"Validation MAE: {best_knn_row['MAE']:.3f}")
print(f"Validation MAPE: {best_knn_row['MAPE (%)']:.3f}%")



### 3.8 Logistic Regression Classification

Logistic regression is a supervised classification algorithm,
rather than a regression method for predicting continuous values.

To apply logistic regression to the diabetes dataset, we
convert the continuous disease-progression target into a
binary classification target.

The median progression value from the training dataset is
used as the classification threshold:

- Class 0: Progression less than or equal to the training median
- Class 1: Progression greater than the training median

The threshold is calculated using only the training target
to prevent data leakage.

We train two logistic regression classifiers using all ten
predictors:

- Model 1: C = 0.1 (stronger regularization)
- Model 2: C = 10.0 (weaker regularization)

Feature scaling is included in each model pipeline.

Unlike the earlier regression models, these classifiers
are evaluated using accuracy, precision, recall, and F1-score.

The validation dataset is used for model selection, while
the test dataset remains held out.


In [ ]:

import importlib
import src.models as models_module

importlib.reload(models_module)

LogisticClassificationModel = (
    models_module.LogisticClassificationModel
)

# Learn the classification threshold from training data only.
progression_threshold = y_train.median()

# Convert continuous targets into binary categories.
y_train_binary = (y_train > progression_threshold).astype(int)
y_val_binary = (y_val > progression_threshold).astype(int)
y_test_binary = (y_test > progression_threshold).astype(int)

print(f"Training median threshold: {progression_threshold:.2f}")
print("\nTraining class distribution:")
print(y_train_binary.value_counts().sort_index())

# Train two logistic regression classifiers.
logistic_models = {}

for C in [0.1, 10.0]:
    model = LogisticClassificationModel(
        C=C,
        random_state=42
    )

    model.fit(X_train, y_train_binary)
    logistic_models[C] = model

    print(f"\nLogistic regression with C={C} trained successfully.")



### 3.9 Logistic Regression Model Evaluation

The two logistic regression classifiers are evaluated using
training and validation data.

Because logistic regression predicts binary categories,
regression metrics such as R², MAE, and MAPE are not appropriate.

Instead, we use the following classification metrics:

- Accuracy: Proportion of correctly classified observations.
- Precision: Proportion of predicted positive cases that are correct.
- Recall: Proportion of actual positive cases correctly identified.
- F1-score: Harmonic mean of precision and recall.

We compare models with C=0.1 and C=10.0.

The best classifier is selected using validation F1-score,
with accuracy considered as an additional metric.

The test dataset remains held out during model selection.


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Evaluate both logistic regression classifiers.
classification_results = []

for C, model in logistic_models.items():
    for dataset_name, X, y in [
        ("Training", X_train, y_train_binary),
        ("Validation", X_val, y_val_binary)
    ]:
        predictions = model.predict(X)

        classification_results.append({
            "C": C,
            "Dataset": dataset_name,
            "Accuracy": accuracy_score(y, predictions),
            "Precision": precision_score(
                y, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y, predictions, zero_division=0
            ),
            "F1-score": f1_score(
                y, predictions, zero_division=0
            )
        })

logistic_comparison = pd.DataFrame(classification_results)

display(
    logistic_comparison
    .sort_values(["C", "Dataset"])
    .round(3)
)


In [ ]:

# Select the classifier using validation F1-score.
logistic_val_results = logistic_comparison[
    logistic_comparison["Dataset"] == "Validation"
]

best_logistic_row = logistic_val_results.sort_values(
    ["F1-score", "Accuracy"],
    ascending=False
).iloc[0]

best_logistic_C = float(best_logistic_row["C"])
best_logistic_model = logistic_models[best_logistic_C]

print(f"Selected logistic regression C: {best_logistic_C}")
print(f"Validation accuracy: {best_logistic_row['Accuracy']:.3f}")
print(f"Validation precision: {best_logistic_row['Precision']:.3f}")
print(f"Validation recall: {best_logistic_row['Recall']:.3f}")
print(f"Validation F1-score: {best_logistic_row['F1-score']:.3f}")



### 3.10 Logistic Regression Findings

The logistic regression classifier with C=0.1 achieved a
training accuracy of 0.730 and validation accuracy of 0.844.

Its validation precision was 0.826, recall was 0.864,
and F1-score was 0.844.

The classifier with C=10.0 achieved a slightly higher
training accuracy of 0.733 but a lower validation
accuracy of 0.822 and F1-score of 0.826.

Both classifiers achieved the same validation recall
of 0.864.

Based on validation F1-score and accuracy, the model
with C=0.1 is selected.

The results suggest that stronger regularization may
help improve generalization for this classification task.

The validation performance is higher than the training
performance, which may reflect differences in difficulty
between the two data splits.

These classification results cannot be directly compared
with the R², MAE, or MAPE values from the regression models
because the prediction tasks are different.


In [ ]:

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Predict validation categories using the selected classifier.
logistic_val_predictions = best_logistic_model.predict(X_val)

# Calculate confusion matrix.
logistic_cm = confusion_matrix(
    y_val_binary,
    logistic_val_predictions,
    labels=[0, 1]
)

# Visualize classification results.
display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=logistic_cm,
    display_labels=["Lower progression", "Higher progression"]
)

display_matrix.plot(cmap="Blues", values_format="d")

plt.title("Logistic Regression Validation Confusion Matrix (C=0.1)")
plt.tight_layout()
plt.show()

print("Confusion matrix:")
print(logistic_cm)



### 3.11 Confusion Matrix Interpretation

The confusion matrix summarizes the predictions made by the
selected logistic regression classifier (C=0.1) on the
validation dataset.

The diagonal cells represent correct classifications,
while the off-diagonal cells represent classification errors.

False positives occur when an observation with lower
progression is predicted to have higher progression.

False negatives occur when an observation with higher
progression is predicted to have lower progression.

The validation recall of 0.864 indicates that the classifier
correctly identifies approximately 86.4% of observations
in the higher-progression category.

The confusion matrix provides additional insight beyond
accuracy by showing the types of classification errors.

The validation confusion matrix produced the following results:

- True negatives: 19
- False positives: 4
- False negatives: 3
- True positives: 19

The classifier correctly predicted 38 out of 45 validation
observations, corresponding to an accuracy of 84.4%.

Of the 22 observations with higher progression, 19 were
correctly identified, giving a recall of 86.4%.

Only three higher-progression observations were incorrectly
classified as lower progression.

Overall, the selected logistic regression model with C=0.1
demonstrates good validation classification performance,
although its reliability on unseen data must still be
assessed using the held-out test set.



### 3.12 Comparison of Selected Regression Models

The best model from each regression family is compared
using validation R², MAE, and MAPE.

The selected models are:

- Multivariate polynomial regression: degree 2
- Decision tree regression: maximum depth 3
- kNN regression: k=15

The model with the highest validation R² is selected
for final testing.

Logistic regression is excluded from this comparison
because it performs binary classification rather than
continuous-value prediction.


In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error
)

# Compare the best models from each regression family.
selected_regression_models = {
    "Polynomial (degree 2)": best_multivariate_model,
    "Decision Tree (depth 3)": best_tree_model,
    "kNN (k=15)": best_knn_model
}

comparison_rows = []

for name, model in selected_regression_models.items():
    predictions = model.predict(X_val)

    comparison_rows.append({
        "Model": name,
        "Validation R²": r2_score(y_val, predictions),
        "Validation MAE": mean_absolute_error(y_val, predictions),
        "Validation MAPE (%)": (
            mean_absolute_percentage_error(y_val, predictions) * 100
        )
    })

final_regression_comparison = pd.DataFrame(comparison_rows)

display(
    final_regression_comparison
    .sort_values("Validation R²", ascending=False)
    .round(3)
)



### 3.13 Final Regression Test Evaluation

The degree 2 multivariate polynomial regression model was
selected because it achieved the highest validation R²
among the regression models evaluated in Part 3.

The selected model is now evaluated on the held-out test
dataset, which has not been used for model training or
selection.

We report test R², MAE, and MAPE to assess how well
the model generalizes to previously unseen observations.


In [ ]:

# Evaluate the selected regression model on the held-out test set.
final_regression_predictions = best_multivariate_model.predict(X_test)

final_regression_test_results = pd.DataFrame([{
    "Model": "Multivariate Polynomial (degree 2)",
    "Test R²": r2_score(y_test, final_regression_predictions),
    "Test MAE": mean_absolute_error(y_test, final_regression_predictions),
    "Test MAPE (%)": (
        mean_absolute_percentage_error(
            y_test, final_regression_predictions
        ) * 100
    )
}])

display(final_regression_test_results.round(3))



### 3.14 Final Regression Test Findings

The selected degree 2 multivariate polynomial regression
model achieved a test R² of 0.337, test MAE of 45.867,
and test MAPE of 39.474%.

The test R² is lower than the validation R² of 0.603,
indicating weaker generalization to the held-out test
observations.

The test MAE also increased from 37.745 to 45.867.

However, the multivariate model outperformed the selected
BMI-only polynomial model from Part 2, which achieved a
test R² of 0.197.

These findings suggest that incorporating all ten
predictors improves predictive performance compared
with using BMI alone, although substantial unexplained
variation remains.

The difference between validation and test performance
also highlights the importance of evaluating models
on data not used during model selection.


### 3.15 Additional Held-Out Model Evaluation

The selected decision tree and kNN configurations are also
evaluated on the held-out test set to complete the comparison
of regression model families.

These configurations were selected using validation data.
Their test results are reported for evaluation only and are
not used to change the original model selections.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import pandas as pd

additional_test_models = {
    "Decision Tree (depth 3)": best_tree_model,
    "kNN (k=15)": best_knn_model
}

additional_test_results = []

for model_name, model in additional_test_models.items():
    predictions = model.predict(X_test)

    additional_test_results.append({
        "Model": model_name,
        "Test R²": r2_score(y_test, predictions),
        "Test MAE": mean_absolute_error(y_test, predictions),
        "Test MAPE (%)": mean_absolute_percentage_error(y_test, predictions) * 100
    })

additional_test_df = pd.DataFrame(additional_test_results)
display(additional_test_df.round(3))

#### 3.15.1 Additional Test Results and Interpretation

The selected decision tree (maximum depth 3) achieved a test R² of 0.308, MAE of 50.371, and MAPE of 44.903%.

The selected kNN model (k=15) achieved a test R² of 0.411, MAE of 46.278, and MAPE of 42.294%.

Although multivariate polynomial regression (degree 2) was selected using the highest validation R², kNN achieved a higher R² on the held-out test set (0.411 versus 0.337).

However, the degree 2 polynomial model achieved slightly lower test MAE (45.867 versus 46.278) and MAPE (39.474% versus 42.294%).

These results demonstrate that model rankings depend on the evaluation dataset and metric. The original model selection remains unchanged because the test set was reserved for final evaluation rather than hyperparameter tuning.


### 3.16 Final Logistic Regression Test Evaluation

The logistic regression classifier with C=0.1 was selected
based on its validation F1-score of 0.844.

We now evaluate the selected classifier on the held-out
test dataset using the binary target created with the
training-set median threshold of 142.

Test accuracy, precision, recall, and F1-score are reported
to assess classification performance on unseen observations.

A confusion matrix is also calculated to identify the
number and types of classification errors.


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Predict binary progression categories on the test dataset.
final_logistic_predictions = best_logistic_model.predict(X_test)

# Evaluate the selected classifier.
final_logistic_test_results = pd.DataFrame([{
    "Model": f"Logistic Regression (C={best_logistic_C})",
    "Test Accuracy": accuracy_score(
        y_test_binary, final_logistic_predictions
    ),
    "Test Precision": precision_score(
        y_test_binary, final_logistic_predictions, zero_division=0
    ),
    "Test Recall": recall_score(
        y_test_binary, final_logistic_predictions, zero_division=0
    ),
    "Test F1-score": f1_score(
        y_test_binary, final_logistic_predictions, zero_division=0
    )
}])

display(final_logistic_test_results.round(3))

# Display test confusion matrix.
print("Test confusion matrix:")
print(confusion_matrix(
    y_test_binary,
    final_logistic_predictions,
    labels=[0, 1]
))



### 3.17 Final Logistic Regression Test Findings

The selected logistic regression classifier with C=0.1
achieved a test accuracy of 0.746, precision of 0.710,
recall of 0.733, and F1-score of 0.721.

The test confusion matrix showed:

- True negatives: 28
- False positives: 9
- False negatives: 8
- True positives: 22

The classifier correctly classified 50 out of 67 test
observations, corresponding to an accuracy of 74.6%.

Test accuracy decreased from the validation accuracy
of 84.4%, while F1-score decreased from 0.844 to 0.721.

This suggests that the validation dataset provided a
somewhat optimistic estimate of classification performance.

Although the classifier performed reasonably well, it
still made 17 errors on the test dataset.

The results demonstrate that logistic regression can
distinguish between lower and higher disease progression
categories, but classification performance is not perfect.



### 3.18 Part 3 Summary and Conclusion

In Part 3, four machine learning model families were
investigated using the diabetes dataset.

**Multivariate Polynomial Regression**

Two polynomial models with degrees 2 and 3 were trained.
The degree 2 model achieved the best validation R² of 0.603.

The degree 3 model severely overfit the training data,
demonstrating that increased complexity does not always
improve generalization.

The selected degree 2 model achieved a test R² of 0.337,
MAE of 45.867, and MAPE of 39.474%.

**Decision Tree Regression**

Two decision trees with maximum depths of 3 and 8 were
evaluated.

The depth 3 tree performed better on validation data,
achieving an R² of 0.413.

The depth 8 tree overfit, achieving a high training R²
but poor validation performance.

**k-Nearest Neighbors Regression**

Two kNN models with k=3 and k=15 were trained.

The k=15 model achieved better validation performance,
with R² of 0.553, compared with 0.323 for k=3.

This demonstrated the importance of selecting an
appropriate neighborhood size.

**Logistic Regression Classification**

Two logistic regression classifiers with C=0.1 and
C=10.0 were trained using a binary target based on
the training-set median disease progression value.

The classifier with C=0.1 achieved the best validation
F1-score of 0.844 and test accuracy of 0.746.

Its test F1-score was 0.721, demonstrating moderate
generalization to unseen observations.

**Overall Conclusion**

Among the regression models evaluated in Part 3,
degree 2 multivariate polynomial regression achieved
the strongest validation performance.

Its test R² of 0.337 exceeded the test R² of 0.197
obtained by the BMI-only polynomial model in Part 2.

Additional held-out testing showed that kNN (k=15) achieved a higher test R² of 0.411 than the selected degree-2 polynomial model's 0.337, although the polynomial model retained lower test MAE and MAPE. The original validation-based selection was not changed.

The experiments demonstrated the importance of
controlling model complexity, comparing training and
validation results, and using a held-out test dataset.

More complex models did not consistently produce
better predictions, and several models showed
substantial overfitting.

Logistic regression addressed a separate binary
classification task and was therefore evaluated
using classification metrics rather than regression
metrics.



## Part 4 — K-Fold Cross-Validation

### 4.1 Cross-Validation Methodology

The previous experiments selected models using a fixed
training, validation, and test split.

Although this approach provides a consistent evaluation
framework, model performance may depend on which
observations are assigned to each subset.

To assess the stability of the regression models,
we perform 5-fold cross-validation.

The cross-validation procedure divides the original
training dataset into five folds. Each model is trained
five times, using four folds for training and the
remaining fold for evaluation.

The mean and standard deviation of R² are reported.

Cross-validation is performed only on the original
training dataset. The validation and test datasets
remain separate to prevent data leakage.

This analysis supplements the original model comparison
rather than replacing the required 75/10/15 split.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

cv_models = {
    f"Polynomial (degree {best_multivariate_degree})":
        best_multivariate_model.pipeline,

    f"Decision Tree (depth {best_tree_depth})":
        best_tree_model.model,

    f"kNN (k={best_knn_k})":
        best_knn_model.pipeline
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in cv_models.items():
    scores = cross_val_score(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring="r2"
    )

    cv_results.append({
        "Model": name,
        "Mean CV R²": scores.mean(),
        "Std CV R²": scores.std()
    })

cv_comparison = pd.DataFrame(cv_results)

display(
    cv_comparison
    .sort_values("Mean CV R²", ascending=False)
    .round(3)
)



### 4.2 Cross-Validation Results and Interpretation

The 5-fold cross-validation results show differences in the
generalization and stability of the selected regression models.

- **kNN (k=15)** achieved the highest mean cross-validation
  R² of 0.432, with a standard deviation of 0.051.
  This indicates relatively consistent performance across
  the five folds.

- **Decision Tree (max_depth=3)** achieved a mean R² of
  0.300 and a standard deviation of 0.053. Its performance
  was relatively consistent but lower than that of kNN.

- **Multivariate Polynomial Regression (degree 2)** achieved
  a mean R² of 0.257 and the highest standard deviation,
  0.165. This suggests greater sensitivity to the
  composition of the training and evaluation folds.

The original validation split favored the degree 2
polynomial model, which achieved a validation R² of 0.603.
However, cross-validation favored kNN, suggesting that
the polynomial model's strong validation performance
may not be representative of its performance across
different data partitions.

These results demonstrate why cross-validation is useful:
it provides a broader assessment of model performance
than a single validation split.

The original model selection and held-out test results
remain unchanged because this analysis is an additional
robustness check, not a replacement for the assignment's
required train-validation-test evaluation procedure.



### 4.3 Cross-Validation Conclusion

The cross-validation experiment identified kNN with
15 neighbors as the strongest of the three evaluated
regression configurations based on mean R².

The degree 2 polynomial model performed best on the
original validation split but showed greater variation
across cross-validation folds.

This highlights an important limitation of relying on
a single validation set for model selection, especially
with a relatively small dataset.

Overall, the cross-validation results provide additional
evidence that kNN may offer more stable predictive
performance, while the polynomial model's results
should be interpreted with greater caution.
